# Lab 01: Brand Voice Lab — Temperature, Top-p and Logprobs

**Module 1: Foundations of Large Language Models**

| | |
|---|---|
| **Estimated time** | 60 minutes |
| **Model** | `gpt-4.1-nano` |
| **Environment** | Course virtual lab (Jupyter), Python 3.10 or later |

### Problem statement

A homeware retailer needs product descriptions that are **accurate**, **consistent with its brand** and **varied enough for different campaigns**. Test generation settings to find the right balance between consistency and creativity.

### Overview

The same model and the same product fact sheet can produce a dependable product page or five fresh campaign variants. The prompt does not change; what changes is **how the model samples its next token**. In this lab, you will call the OpenAI API from a Jupyter notebook and generate product descriptions from the retailer's **product fact sheets** and **brand voice guide**. You will test how **temperature** and **top-p** trade consistency for variety, check whether more variety costs accuracy, and use **token log probabilities (logprobs)** to see how confident the model is in a decision.

### Learning objectives

By the end of this lab, you will be able to:

- Explain how temperature and top-p change the next-token probability distribution.
- Choose appropriate settings for hyperparameters in the API to address different use cases.
- Use logprobs to evaluate model confidence in output text.
- Compare repeated sampling with logprobs as two signals of model confidence.

### Getting started

- **Setup:** if this is your first lab, open `START_HERE.ipynb` in the repo root and run all its cells once. It installs the packages and saves the course API key to the repo's `.env` file, which every lab reads. Before each session, run `python sync.py` (or the first cell of `START_HERE.ipynb`) to get the latest version of the labs.
- **Time:** about 60 minutes. Run the cells in order, top to bottom.
- **How to work:** each task is broken into numbered steps. Steps tell you what to run, what to look for and what to write down. Cells marked `# DO NOT EDIT` are shared code; change only the lines a step asks you to change.
- **Cost:** every call uses `gpt-4.1-nano` with short outputs. The whole lab costs a few cents.

### Tasks

1. Set up the environment and load the product fact sheets
2. Compare descriptions across temperature settings
3. Restrict sampling with top-p
4. Read model confidence with logprobs
5. Compare repeated sampling with logprobs
6. Choose settings for product pages and campaigns

---
## Task 1: Set up the environment and load the product fact sheets (~10 min)

### 1.1 Check your API key

Your API key lives in the `.env` file at the root of the repo, saved there by `START_HERE.ipynb`. If you have not run `START_HERE.ipynb` yet, do that first.

Never paste your key into the notebook itself. Notebooks are easy to share by accident.

### 1.2 Check the packages

The packages were installed by `sync.py`. Run the cell below to confirm. If you see `ModuleNotFoundError`, uncomment the `%pip` line, run the cell, then restart the kernel.

In [ ]:
# %pip install -q -r ../requirements.txt   # only needed if the imports below fail

import openai, dotenv, matplotlib
print("openai", openai.__version__, "| matplotlib", matplotlib.__version__)

### 1.3 Connect to the API and make a test call

Run the next two cells. The first loads your key from `.env`, creates the API client and sets the model. The second sends a one-line test prompt.

**Expected result:** a one-sentence greeting. If you see an authentication error, re-run the key cell in `START_HERE.ipynb`, then restart the kernel.

In [ ]:
import math, re
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True), override=True)   # finds the .env file at the repo root
client = OpenAI(max_retries=5)   # the class shares one key, so retry briefly if rate-limited

MODEL = "gpt-4.1-nano"

In [ ]:
# Test that the API is working
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in one sentence."}],
    max_completion_tokens=30,
)
print(response.choices[0].message.content)

### 1.4 Read the brand voice guide and product fact sheets

Every description in this lab is generated from the same two inputs: the retailer's **brand voice guide** and a **product fact sheet**. Run the cell and read both.

Keep two definitions in mind for the rest of the lab:
- A description is **accurate** if every claim in it comes from the fact sheet.
- A description is **on-brand** if it follows the guide.

In [ ]:
BRAND_VOICE = """BRAND VOICE GUIDE — Linden & Loam (homewares)
- Tone: warm, calm and understated. Write like a knowledgeable friend, not a salesperson.
- Use plain words and short sentences.
- Name real materials and how the product is made.
- No exclamation marks, no emojis, no ALL CAPS.
- No hype or superlatives such as "best", "perfect", "stunning" or "must-have".
- Never claim anything that is not in the product fact sheet."""

FACT_SHEETS = {
    "lamp": {
        "text": """PRODUCT FACT SHEET — Mira Table Lamp
- Base: hand-glazed stoneware, made in Portugal; the glaze varies slightly from piece to piece
- Shade: natural linen, 30 cm diameter
- Height: 42 cm
- Bulb: E27 fitting, max 40 W, bulb not included
- Colours: oat, charcoal
- Care: wipe the base with a dry cloth""",
        "key_facts": ["stoneware", "linen", "Portugal"],
    },
    "duvet": {
        "text": """PRODUCT FACT SHEET — Stonewashed Linen Duvet Cover
- Material: 100% European flax linen, stonewashed for softness
- Sizes: Double 200 x 200 cm, King 230 x 220 cm
- Closure: coconut-shell buttons
- Colours: sage, clay, white
- Care: machine wash at 40 degrees C; gets softer with every wash
- Pillowcases sold separately""",
        "key_facts": ["linen", "stonewashed", "buttons"],
    },
}

print(BRAND_VOICE, "\n")
for sheet in FACT_SHEETS.values():
    print(sheet["text"], "\n")

### 1.5 Load the helper functions

Run the cell below. It defines the functions every task uses. **Do not modify it.**

- `generate()` sends one prompt and exposes the hyperparameters you will test. As in the lecture example, `max_completion_tokens` caps the response length, and `temperature` and `top_p` control sampling.
- `description_prompt()` builds the prompt that asks for a product description from a fact sheet and the brand voice guide.
- `check_facts()` and `show_description()` print each description with a quick accuracy check. The check lists key facts the description left out and any **numbers that do not appear in the fact sheet**, which often signal an invented detail. It has two blind spots: it **cannot catch invented claims without numbers** (for example, "lasts for years" or "hypoallergenic"), and it **cannot judge brand voice**. Always read the output yourself.
- `first_token_distribution()` reads the logprobs of the first generated token. You will use it in Task 4.

In [ ]:
# DO NOT EDIT — shared helper functions for all tasks

def generate(prompt, temperature=1.0, top_p=1.0, max_completion_tokens=100,
             logprobs=False, top_logprobs=None):
    """Call the Chat Completions API and return the full response object."""
    kwargs = dict(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=max_completion_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    if logprobs:
        kwargs["logprobs"] = True
        if top_logprobs is not None:
            kwargs["top_logprobs"] = top_logprobs
    return client.chat.completions.create(**kwargs)


def description_prompt(product, sentences=2):
    return f"""{BRAND_VOICE}

{FACT_SHEETS[product]["text"]}

Write a {sentences}-sentence product description for this item, following the brand voice guide.
Use only facts from the fact sheet."""


def check_facts(text, product):
    """Return (key facts missing, numbers not found in the fact sheet)."""
    sheet = FACT_SHEETS[product]
    missing = [f for f in sheet["key_facts"] if f.lower() not in text.lower()]
    sheet_numbers = set(re.findall(r"\d+(?:\.\d+)?", sheet["text"]))
    unsupported = sorted(set(re.findall(r"\d+(?:\.\d+)?", text)) - sheet_numbers)
    return missing, unsupported


def show_description(text, product):
    missing, unsupported = check_facts(text, product)
    print(f"    {text}")
    print(f"    [fact check] key facts not mentioned: {missing or 'none'} | "
          f"numbers not in fact sheet: {unsupported or 'none'}")


def first_token_distribution(resp):
    """Return the generated first token and its top candidates as (token, probability) pairs."""
    first = resp.choices[0].logprobs.content[0]
    candidates = [(c.token, math.exp(c.logprob)) for c in first.top_logprobs]
    return first.token, candidates

---
## Task 2: Compare descriptions across temperature settings (~10 min)

At every step, the model produces a score (logit) for each possible next token. **Temperature** divides those scores by a value *T* before they are turned into probabilities:

- **Low temperature** (close to 0): the distribution becomes more peaked on the highest-probability tokens, so output is close to deterministic. At `temperature=0`, the model picks the most likely token every time (greedy decoding).
- **High temperature** (above 1): the distribution flattens, so less likely tokens are picked more often. Output becomes more varied and, eventually, less reliable.

### 2.1 Generate descriptions at five temperatures

Run the cell. It asks for a description of the Mira Table Lamp 3 times at each temperature: **0, 0.3, 0.7, 1.0 and 1.5**. Only the temperature changes; the prompt and fact sheet stay the same. Each description is followed by its fact-check line.

In [ ]:
product = "lamp"
temperatures = [0, 0.3, 0.7, 1.0, 1.5]
num_runs = 3

for temp in temperatures:
    print(f"\n{'='*70}\nTEMPERATURE = {temp}\n{'='*70}")
    for run in range(1, num_runs + 1):
        resp = generate(description_prompt(product), temperature=temp, max_completion_tokens=120)
        print(f"\n  Run {run}:")
        show_description(resp.choices[0].message.content, product)

### 2.2 Inspect the outputs

Work through the output from top to bottom:
- **Within each temperature**, compare the 3 runs. Are they word-for-word the same, slightly reworded or clearly different?
- **Across temperatures**, find the point where the descriptions start to change in structure, not just in word choice.
- **At the highest temperatures**, read each description against the fact sheet and the brand voice guide. Note any claim that is not in the fact sheet, whether or not the fact check flagged it.

### 2.3 ✍️ Record your observations

Answer in 2–3 sentences each:

1. At `temperature=0`, how similar were the 3 descriptions? Why?
2. At which temperature did you first notice clearly different descriptions across runs?
3. At `temperature=1.5`, were the descriptions still **accurate** and **on-brand**? Did the fact check flag anything, and did you spot any invented detail it missed?

> *Your answer here*

---
## Task 3: Restrict sampling with top-p (~10 min)

**Top-p** (nucleus sampling) controls the **size of the candidate token set**. The model sorts tokens by probability and keeps only the smallest set whose cumulative probability reaches `top_p`, then samples from that set.

- `top_p=0.1` → only the tokens that make up the top 10% of probability mass (very conservative)
- `top_p=1.0` → all tokens (no truncation; this is the default)

Temperature changes **how likely** each token is. Top-p changes **which tokens are allowed** at all. The lecture treats them as **alternatives**: adjust one and leave the other at its default.

### 3.1 Generate descriptions at three top-p values

Run the cell. It keeps `temperature=1.0` (the default) and changes only `top_p`: **0.1, 0.5 and 1.0**, with 3 descriptions at each setting.

In [ ]:
product = "lamp"
top_p_values = [0.1, 0.5, 1.0]
num_runs = 3

for tp in top_p_values:
    print(f"\n{'='*70}\nTOP_P = {tp}  (temperature fixed at 1.0)\n{'='*70}")
    for run in range(1, num_runs + 1):
        resp = generate(description_prompt(product), temperature=1.0, top_p=tp,
                        max_completion_tokens=120)
        print(f"\n  Run {run}:")
        show_description(resp.choices[0].message.content, product)

### 3.2 Compare with Task 2

Scroll up and put your Task 2 and Task 3 outputs side by side:
- Compare `top_p=0.1` with `temperature=0`. Both are "conservative", but are the 3 runs equally repetitive?
- Compare `top_p=1.0` here with `temperature=1.0` in Task 2. These are the same settings, so the outputs should look similar in variety.
- Check the fact-check lines. Did a low top-p reduce the invented details you saw at high temperature?

### 3.3 ✍️ Record your observations

1. How did `top_p=0.1` compare to `temperature=0` from Task 2? Were the descriptions equally repetitive? Why might they differ?
2. Did lowering top-p reduce the problems you saw at high temperature in Task 2, such as invented details?
3. Both settings can make output more conservative. Why does the lecture recommend changing only one of them at a time?

> *Your answer here*

---
## Task 4: Read model confidence with logprobs (~12 min)

Before choosing a token, the model assigns a probability to every possible next token. Setting `logprobs=True` returns those probabilities in log form for the tokens it generated, and `top_logprobs` adds the most likely alternatives at each position.

- `probability = e^logprob`, so a **logprob closer to 0** means a higher probability.
- If one token has almost all the probability, the model is **confident** in that choice. If probability is split between several tokens, it is **unsure**.

> **Important:** Confidence is not correctness. Logprobs show how strongly the model prefers an answer, not whether the answer is right. A model can be confidently wrong.

In this task, the retailer asks the model to check descriptions against the brand voice guide before they go live, answering **Yes** or **No**. The answer alone tells you **what** the model decided. Logprobs tell you **how strongly** it preferred that answer.

### 4.1 Read one logprob yourself

Start with a single call so you can see where the numbers come from. The cell asks the model to check one on-brand description. Fill in the two blanks marked `___`:

1. The **first** generated token is at position `0` of `resp.choices[0].logprobs.content`.
2. Convert its logprob into a probability with `math.exp()`.

**Expected result:** usually a first token of `'Yes'` with a probability well above 50%. Because this call samples at `temperature=1.0`, you may occasionally see `'No'`; if so, run the cell again and compare.

In [ ]:
# YOUR TURN — fill in the two blanks marked ___
resp = generate(f"""{BRAND_VOICE}

Description to check:
"The Mira Table Lamp pairs a hand-glazed stoneware base with a natural linen shade."

Does this description follow the brand voice guide? Answer with exactly one word: Yes or No.""",
                temperature=1.0, max_completion_tokens=2, logprobs=True, top_logprobs=5)

first = resp.choices[0].logprobs.content[___]   # position of the first generated token
probability = math.exp(___)                     # convert this token's logprob to a probability

print(f"First token: {first.token!r} | logprob: {first.logprob:.4f} | probability: {probability:.2%}")

### 4.2 Compare three descriptions

Read the three descriptions in the cell below and predict which will get a strong **Yes**, which a strong **No**, and which is borderline. Then run the cell. It checks each description once and shows:
- `P(Yes)` and `P(No)`: the probability of each answer at the first generated token
- `Answer`: what the model actually wrote
- a bar chart of the two probabilities for each description

The cell uses `temperature=1.0`, the same setting as the first sampling run in Task 5, so you can compare the two directly. Because the answer is sampled, `Answer` can occasionally differ from the more likely label. That is expected.

> **Note:** If an answer is not among the returned candidates, it shows as `0.00%`. That means it was not in the top 10, not that its probability is exactly zero.

In [ ]:
# DO NOT EDIT — run as provided, then inspect the output
import matplotlib.pyplot as plt

candidates_to_check = [
    "The Mira Table Lamp pairs a hand-glazed stoneware base, made in Portugal, with a natural linen "
    "shade. Because each base is glazed by hand, the colour varies slightly from lamp to lamp.",
    "OMG!!! The BEST lamp you will EVER own. This stunning beauty will totally transform your space!",
    "The Mira Table Lamp casts a soft, warm glow. Its hand-glazed stoneware base and natural linen "
    "shade are made to last for years.",
]
LABELS = ["Yes", "No"]

def brand_check_prompt(description):
    return f"""{BRAND_VOICE}

Description to check:
"{description}"

Does this description follow the brand voice guide? Answer with exactly one word: Yes or No."""

task4_results = []
print(f"{'#':<3} {'Description (truncated)':<48} {'P(Yes)':>9} {'P(No)':>9}  Answer")
print("-" * 82)
for n, description in enumerate(candidates_to_check, start=1):
    resp = generate(brand_check_prompt(description), temperature=1.0, max_completion_tokens=2,
                    logprobs=True, top_logprobs=10)
    _, candidates = first_token_distribution(resp)
    probs = {label: 0.0 for label in LABELS}
    for token, p in candidates:
        label = token.strip().capitalize()   # count 'Yes', ' Yes' and 'yes' as the same answer
        if label in probs:
            probs[label] += p
    answer = resp.choices[0].message.content.strip()
    task4_results.append({"description": description, **probs})
    short = description[:45] + "..." if len(description) > 45 else description
    print(f"{n:<3} {short:<48} {probs['Yes']:>9.2%} {probs['No']:>9.2%}  {answer}")

# Chart: one pair of bars per description
x = range(len(task4_results))
plt.figure(figsize=(8, 3.5))
plt.bar([i - 0.2 for i in x], [r["Yes"] for r in task4_results], width=0.4, label="P(Yes)", color="steelblue")
plt.bar([i + 0.2 for i in x], [r["No"] for r in task4_results], width=0.4, label="P(No)", color="coral")
plt.xticks(list(x), [f"Description {i + 1}" for i in x])
plt.ylabel("Probability")
plt.ylim(0, 1.05)
plt.legend()
plt.title("Brand check: first-token probabilities")
plt.tight_layout()
plt.show()

### 4.3 Inspect the probability gaps

For each description, look at the **gap** between `P(Yes)` and `P(No)`, not just the answer:
- A large gap means the model was confident.
- A small gap means it was unsure, even if it still wrote Yes or No.

Then read description 3 against the **fact sheet** from Task 1. It has no hype words. Is everything it claims actually in the fact sheet?

### 4.4 ✍️ Record your observations

1. For the clearly on-brand and clearly off-brand descriptions, how large was the gap between `P(Yes)` and `P(No)`? What does that tell you about the model's confidence?
2. Was the gap smaller for the third description? Which rule in the guide does it break? Would `check_facts()` have caught it?
3. If you were checking hundreds of descriptions, how would you use the probabilities, not just the Yes/No answer, to decide which ones a person should review?
4. The labels here are single tokens. What would go wrong if they were longer, such as `On-brand` and `Off-brand`, and were split into several tokens?

> *Your answer here*

---
## Task 5: Compare repeated sampling with logprobs (~10 min)

Logprobs show the model's preference from the **inside**. You can also observe it from the **outside**: send the same prompt many times with sampling turned on and count how often each answer comes back.

Temperature connects the two. At `temperature=1.0`, the model samples in proportion to the probabilities, so an answer with `P = 0.70` should come back about 70% of the time. A lower temperature sharpens the distribution, so the more likely answer wins even more often.

### 5.1 Make a prediction

Using your Task 4 table, write down for each description how many of 20 samples you expect to be **Yes** at `temperature=1.0`, and whether that number will go up or down at `temperature=0.3`.

### 5.2 Sample each description 20 times

Run the cell. It sends the same brand-check prompt 20 times per description at each temperature (120 short calls, about a minute) and counts the Yes answers. The first column repeats `P(Yes)` from Task 4 so you can compare.

In [ ]:
# DO NOT EDIT — run after Task 4
N_SAMPLES = 20
SAMPLE_TEMPERATURES = [1.0, 0.3]

def first_word_label(text):
    """Map a reply to Yes, No or Other using its first word."""
    words = (text or "").strip().split()
    word = words[0].strip(".,!:;\"'").capitalize() if words else ""
    return word if word in LABELS else "Other"

print(f"{'#':<3} {'P(Yes) from Task 4':>19}   {'Yes in 20 samples, T=1.0':>25}   {'T=0.3':>6}")
print("-" * 62)
for n, row in enumerate(task4_results, start=1):
    yes_counts = []
    for temp in SAMPLE_TEMPERATURES:
        counts = {"Yes": 0, "No": 0, "Other": 0}
        for _ in range(N_SAMPLES):
            reply = generate(brand_check_prompt(row["description"]), temperature=temp,
                             max_completion_tokens=2).choices[0].message.content
            counts[first_word_label(reply)] += 1
        yes_counts.append(counts["Yes"])
    print(f"{n:<3} {row['Yes']:>19.0%}   {yes_counts[0]:>18} / {N_SAMPLES}   {yes_counts[1]:>3} / {N_SAMPLES}")

### 5.3 Compare the counts with your prediction and with Task 4

- Divide each `T=1.0` count by 20 and compare it with `P(Yes)`. With only 20 samples, expect them to be close, not identical.
- Compare the `T=1.0` and `T=0.3` columns. For which description did the count change the most?
- Check your prediction from 5.1. Where were you wrong, and why?

### 5.4 ✍️ Record your observations

1. At `temperature=1.0`, how close was the share of Yes samples to `P(Yes)` from Task 4? Which description varied the most across samples?
2. What happened at `temperature=0.3`? Explain why, using how temperature reshapes the distribution.
3. Both methods estimate confidence. When would you use repeated sampling instead of logprobs? Consider cost, and whether the API you use returns logprobs at all.

> *Your answer here*

---
## Task 6: Choose settings for product pages and campaigns (~10 min)

Now find the balance the problem statement asks for. The retailer has two jobs for the linen duvet cover:

- **Product page description:** shown on the website, so it must be **accurate and consistent**. Every regeneration should say essentially the same thing.
- **Campaign variants:** five different short descriptions for an email campaign, so they must be **varied**, while still accurate and on-brand.

Following the lecture's rule, **keep `top_p=1.0` and adjust only `temperature`**. There is no single correct value; choose one for each job and explain your reasoning.

### 6.1 Choose your temperatures

Use your Task 2 results as a guide:
- **Product page:** start low, somewhere between `0` and `0.3`.
- **Campaign variants:** start in the middle, somewhere between `0.7` and `1.0`, then adjust based on what you see.

Edit only these two values in the code below:
```python
PAGE_TEMP = ___
CAMPAIGN_TEMP = ___
```

### 6.2 Generate the product page description

Enter `PAGE_TEMP` in the cell below and run it. It generates the same product page description 3 times. Then check:
- Do all three runs say essentially the same thing?
- Did the fact check flag anything? Did you spot an invented claim it missed?
- Are they on-brand?

In [ ]:
# TODO: fill in your chosen temperature (top_p stays at the default 1.0)
PAGE_TEMP = ___

print("=== Product page description (3 runs) ===\n")
for run in range(1, 4):
    resp = generate(description_prompt("duvet", sentences=3), temperature=PAGE_TEMP,
                    top_p=1.0, max_completion_tokens=150)
    print(f"Run {run}:")
    show_description(resp.choices[0].message.content, "duvet")
    print()

### 6.3 Generate the campaign variants

Enter `CAMPAIGN_TEMP` in the cell below and run it. It generates 5 one-sentence campaign variants. Then check:
- Are the variants noticeably different from one another?
- Do they stay accurate and on-brand? Read them yourself; the fact check cannot judge tone.
- Is the variation too small, about right or excessive? If needed, change `CAMPAIGN_TEMP` and run again.

In [ ]:
# TODO: fill in your chosen temperature (top_p stays at the default 1.0)
CAMPAIGN_TEMP = ___

print("=== Campaign variants (5 runs) ===\n")
for run in range(1, 6):
    resp = generate(description_prompt("duvet", sentences=1), temperature=CAMPAIGN_TEMP,
                    top_p=1.0, max_completion_tokens=60)
    print(f"Variant {run}:")
    show_description(resp.choices[0].message.content, "duvet")
    print()

### 6.4 ✍️ Record your observations

1. **Settings:** What temperature did you choose for each job, and why does it fit the requirement?
2. **Consistency:** Did the product page runs stay consistent? If not, what changed?
3. **Variety vs. accuracy:** Were the campaign variants varied enough without inventing details or breaking the brand voice? Did you change your first value, and at what point did more variety start to cost accuracy?
4. **Practical rule:** Write one rule you would give the marketing team for choosing the temperature for a new product line.

> *Your answer here*

---
## Discussion questions (optional)

If time permits, discuss the following with your partner or group:

1. `temperature=0` produces nearly identical outputs across runs. Why is this useful for a product page but a problem for campaign variants? Can you think of a scenario where deterministic output is actually harmful?

2. Based on the lecture and your experiments in Tasks 2 and 3, explain in your own words how temperature and top-p differ, and why practitioners change only one of them.

3. In Task 4, suppose you had only the Yes/No answer and no logprobs. What would you know about the model's confidence? What do logprobs add, and what are their limits?

4. In Task 5, how do temperature and logprobs together decide which answer gets picked? What does this suggest about running the same check several times and taking the majority answer?

5. The GPT-3 paper (Brown et al., 2020) showed that model performance scales with size. Do you think inference-time hyperparameters such as temperature and top-p become more or less important as models get larger and more capable? Why?